# vLLM Benchmark Results Analysis

This notebook loads all CSV files produced by `run_experiment.sh` and visualizes:
- TTFT, TPOT, E2E latency distributions (box plots + violin plots)
- Throughput vs latency Pareto frontier per experiment
- Per-experiment percentile tables (P50, P90, P95, P99)
- Cost per token estimation
- Comparison heatmap across all experiments

In [ ]:
import glob
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid', palette='tab10')
plt.rcParams['figure.dpi'] = 120

# ─── Configuration ────────────────────────────────────────────────────────────
RESULTS_ROOT = Path('results')   # relative to this notebook

# If you want to analyze a specific run, set RUN_ID.
# Leave as None to auto-pick the most recent run.
RUN_ID = None

# GPU cost on GKE (L4 on-demand ≈ $0.70/hr — update as needed)
GPU_HOURLY_COST_USD = 0.70
# ──────────────────────────────────────────────────────────────────────────────

In [ ]:
# ─── Load data ────────────────────────────────────────────────────────────────
if RUN_ID is None:
    runs = sorted(RESULTS_ROOT.glob('run_*'))
    if not runs:
        raise FileNotFoundError(f'No run_* directories found under {RESULTS_ROOT}. Run run_experiment.sh first.')
    run_dir = runs[-1]
    RUN_ID = run_dir.name
else:
    run_dir = RESULTS_ROOT / RUN_ID

print(f'Loading results from: {run_dir}')

frames = []
for csv_path in sorted(run_dir.rglob('*custom_metrics*.csv')):
    # Infer experiment name and concurrency from path
    parts = csv_path.stem.split('__')
    exp_name   = csv_path.parent.name
    users      = int(parts[1].replace('u', '')) if len(parts) > 1 else 0
    prompt_cat = parts[2] if len(parts) > 2 else 'all'

    df = pd.read_csv(csv_path)
    df['experiment']   = exp_name
    df['users']        = users
    df['prompt_cat_file'] = prompt_cat
    frames.append(df)

if not frames:
    raise FileNotFoundError(f'No custom_metrics CSV files found in {run_dir}')

df_all = pd.concat(frames, ignore_index=True)
df_ok  = df_all[df_all['success'] == True].copy()

print(f'Total requests loaded : {len(df_all):,}')
print(f'Successful            : {len(df_ok):,}  ({100*len(df_ok)/len(df_all):.1f}%)')
print(f'Experiments           : {df_ok["experiment"].nunique()}')
df_ok.head(3)

In [ ]:
# ─── Percentile summary table ─────────────────────────────────────────────────
def pct(s, p): return s.quantile(p / 100)

summary = (
    df_ok
    .groupby(['experiment', 'users', 'category'])
    .agg(
        n               = ('ttft_ms', 'count'),
        ttft_mean       = ('ttft_ms', 'mean'),
        ttft_p50        = ('ttft_ms', lambda s: pct(s, 50)),
        ttft_p95        = ('ttft_ms', lambda s: pct(s, 95)),
        ttft_p99        = ('ttft_ms', lambda s: pct(s, 99)),
        tpot_mean       = ('tpot_ms', 'mean'),
        tpot_p50        = ('tpot_ms', lambda s: pct(s, 50)),
        tpot_p99        = ('tpot_ms', lambda s: pct(s, 99)),
        e2e_mean        = ('e2e_ms',  'mean'),
        e2e_p50         = ('e2e_ms',  lambda s: pct(s, 50)),
        e2e_p99         = ('e2e_ms',  lambda s: pct(s, 99)),
        output_tokens   = ('output_tokens', 'sum'),
    )
    .reset_index()
)

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.1f}'.format)
summary

In [ ]:
# ─── TTFT box plot — by experiment, faceted by concurrency ────────────────────
fig, axes = plt.subplots(1, df_ok['users'].nunique(), figsize=(5 * df_ok['users'].nunique(), 5), sharey=False)
if df_ok['users'].nunique() == 1:
    axes = [axes]

for ax, u in zip(axes, sorted(df_ok['users'].unique())):
    subset = df_ok[df_ok['users'] == u]
    sns.boxplot(data=subset, x='experiment', y='ttft_ms', hue='category', ax=ax,
                flierprops=dict(marker='o', markersize=2, alpha=0.4))
    ax.set_title(f'TTFT (ms) — {u} concurrent user(s)')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=40)
    ax.set_ylabel('TTFT (ms)')

plt.tight_layout()
plt.savefig(run_dir / 'ttft_boxplot.png', bbox_inches='tight')
plt.show()

In [ ]:
# ─── TPOT box plot ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, df_ok['users'].nunique(), figsize=(5 * df_ok['users'].nunique(), 5), sharey=False)
if df_ok['users'].nunique() == 1:
    axes = [axes]

for ax, u in zip(axes, sorted(df_ok['users'].unique())):
    subset = df_ok[df_ok['users'] == u]
    sns.boxplot(data=subset, x='experiment', y='tpot_ms', hue='category', ax=ax,
                flierprops=dict(marker='o', markersize=2, alpha=0.4))
    ax.set_title(f'TPOT (ms/token) — {u} concurrent user(s)')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=40)
    ax.set_ylabel('TPOT (ms/token)')

plt.tight_layout()
plt.savefig(run_dir / 'tpot_boxplot.png', bbox_inches='tight')
plt.show()

In [ ]:
# ─── Throughput vs P95 TTFT — Pareto frontier chart ──────────────────────────
# For each (experiment, users) compute: output token throughput & P95 TTFT

def throughput_per_run(grp):
    if grp.empty:
        return pd.Series({'token_throughput': 0, 'ttft_p95': None})
    total_tokens = grp['output_tokens'].sum()
    # Approximate test duration from timestamp spread
    duration_s = grp['timestamp'].max() - grp['timestamp'].min()
    if duration_s < 1:
        duration_s = 1
    return pd.Series({
        'token_throughput': total_tokens / duration_s,
        'ttft_p95': grp['ttft_ms'].quantile(0.95),
    })

pareto = (
    df_ok.groupby(['experiment', 'users'])
    .apply(throughput_per_run)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 6))
experiments = pareto['experiment'].unique()
palette = sns.color_palette('tab10', len(experiments))

for exp, color in zip(experiments, palette):
    subset = pareto[pareto['experiment'] == exp].sort_values('users')
    ax.plot(subset['ttft_p95'], subset['token_throughput'],
            marker='o', label=exp, color=color, linewidth=1.8)
    for _, row in subset.iterrows():
        ax.annotate(f'u={int(row["users"])}',
                    (row['ttft_p95'], row['token_throughput']),
                    textcoords='offset points', xytext=(4, 4), fontsize=7, color=color)

ax.set_xlabel('P95 TTFT (ms)  [lower is better →]')
ax.set_ylabel('Output Token Throughput (tokens/s)  [higher is better ↑]')
ax.set_title('Throughput vs Latency — Pareto Frontier')
ax.legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.savefig(run_dir / 'pareto_frontier.png', bbox_inches='tight')
plt.show()

In [ ]:
# ─── Cost per token ───────────────────────────────────────────────────────────
# For each experiment, estimate cost per million tokens

cost_rows = []
for (exp, users), grp in df_ok.groupby(['experiment', 'users']):
    total_tokens  = grp['output_tokens'].sum()
    duration_hr   = (grp['timestamp'].max() - grp['timestamp'].min()) / 3600
    if duration_hr < 1e-6 or total_tokens == 0:
        continue
    cost_per_m    = (GPU_HOURLY_COST_USD * duration_hr) / (total_tokens / 1_000_000)
    cost_rows.append({'experiment': exp, 'users': users,
                      'total_tokens': total_tokens, 'cost_per_M_tokens': cost_per_m})

cost_df = pd.DataFrame(cost_rows).sort_values(['experiment', 'users'])

fig, ax = plt.subplots(figsize=(10, 5))
for exp in cost_df['experiment'].unique():
    sub = cost_df[cost_df['experiment'] == exp]
    ax.plot(sub['users'], sub['cost_per_M_tokens'], marker='o', label=exp)

ax.set_xlabel('Concurrent Users')
ax.set_ylabel('Cost per Million Output Tokens (USD)')
ax.set_title(f'Cost per Token — GPU hourly rate: ${GPU_HOURLY_COST_USD}/hr')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(run_dir / 'cost_per_token.png', bbox_inches='tight')
plt.show()

cost_df

In [ ]:
# ─── Heatmap: P99 TTFT across experiments × concurrency ─────────────────────
pivot = summary.pivot_table(
    index='experiment', columns='users', values='ttft_p99', aggfunc='mean'
)

fig, ax = plt.subplots(figsize=(max(6, pivot.shape[1] * 1.5), max(4, pivot.shape[0] * 0.7)))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='RdYlGn_r',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'P99 TTFT (ms)'})
ax.set_title('P99 TTFT (ms) — lower is better')
ax.set_xlabel('Concurrent Users')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig(run_dir / 'heatmap_p99_ttft.png', bbox_inches='tight')
plt.show()

In [ ]:
# ─── Error analysis ───────────────────────────────────────────────────────────
errors = df_all[df_all['success'] == False]
print(f'Total failed requests: {len(errors)}')

if not errors.empty:
    print('\nError breakdown:')
    display(errors.groupby(['experiment', 'error_type']).size().rename('count').reset_index())

    fig, ax = plt.subplots(figsize=(8, 4))
    errors.groupby('error_type').size().sort_values().plot(kind='barh', ax=ax, color='salmon')
    ax.set_title('Failed Requests by Error Type')
    ax.set_xlabel('Count')
    plt.tight_layout()
    plt.show()
else:
    print('✅  No errors!')

In [ ]:
# ─── Export full summary to CSV ───────────────────────────────────────────────
out = run_dir / 'summary.csv'
summary.to_csv(out, index=False)
print(f'Summary saved to: {out}')